# LegalHalluLens Debate Refinement With LangGraph



This notebook starts from an existing clause-extraction artifact and rebuilds the debate stage as a fresh LangGraph workflow.



## Graph design



Contract-level graph:

- `prepare_contract_rows`

- `dispatch_debate_workers`

- `assemble_contract_output`

- `evaluate_contract`

- `save_outputs`



Clause-level graph:

- `retriever`

- `skeptic`

- `supporter`

- `arbiter_policy`

- `finalize_clause`



## Goals



- start from previously extracted `answer_ai` and `is_impossible_ai` values

- debate only the extracted artifact, not rerun extraction

- keep contract-only evidence, explicit routing, and auditable logs

- save fresh LangGraph outputs for inspection and scaling


## Setup

Install dependencies if needed: `%pip install langgraph langchain-core`

LLM wrappers (`utils/llm.py`, `utils/llm_ollama.py`), clause definitions, and judge logic are provided by the repo.

In [ ]:

import json
import operator
import os
import pickle
import re
import sys
import time
import traceback
import importlib
import uuid
from copy import deepcopy
from datetime import datetime
from pathlib import Path
from typing import Annotated, Any, Dict, List, Literal, Optional
from typing_extensions import TypedDict

import numpy as np
import pandas as pd

project_root = Path.cwd().parent if 'Notebooks' in str(Path.cwd()) else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import utils.llm_ollama
importlib.reload(utils.llm_ollama)

from utils.helpers import download_parse_json, parse_json_response
from utils.llm import call_llm, count_tokens, get_model_name
from utils.llm_func import judge_semantic_match
from utils.clauses_prompts import CLAUSE_DEFINITIONS, CLAUSE_TO_TYPE
# from utils.llm_ollama import call_llm
try:
    from langgraph.graph import StateGraph, START, END
    from langgraph.checkpoint.memory import InMemorySaver
except ImportError as error:
    raise ImportError(
        "LangGraph is not installed. Install with: %pip install langgraph langchain-core"
    ) from error

# ── Model config ──────────────────────────────────────────────────────────────
EXTRACTOR_MODEL   = "google/gemma-4-26b-a4b-it"
SKEPTIC_MODEL     = EXTRACTOR_MODEL
SUPPORTER_MODEL   = EXTRACTOR_MODEL
JUDGE_MODEL       = EXTRACTOR_MODEL
ADJUDICATOR_MODEL = EXTRACTOR_MODEL

SKEPTIC_CLAIM_TYPES = None
DEBATE_ROUNDS                = 2
MAX_EXTRACTION_RETRIES       = 2
DEBATE_MAX_WORKERS           = 8
NUM_CONTRACTS                = 5
NUM_RUNS                     = 1
EXTRACTION_CHECKPOINT_INTERVAL = 10

# ── Directories ───────────────────────────────────────────────────────────────
BASE_DIR       = Path("./debate_results_langgraph")
CHECKPOINT_DIR = BASE_DIR / "checkpoints"
EXTRACTION_DIR = BASE_DIR / "extractions"
JUDGE_DIR      = BASE_DIR / "judge_results"
SUMMARY_DIR    = BASE_DIR / "summaries"
DEBATE_LOG_DIR = BASE_DIR / "debate_logs"
ANALYSIS_DIR   = BASE_DIR / "analysis"

for path in [BASE_DIR, CHECKPOINT_DIR, EXTRACTION_DIR, JUDGE_DIR, SUMMARY_DIR, DEBATE_LOG_DIR, ANALYSIS_DIR]:
    path.mkdir(exist_ok=True)

print("Environment ready")
print("Extractor    :", get_model_name(EXTRACTOR_MODEL))
print("Agent A (Yes):", get_model_name(SUPPORTER_MODEL))
print("Agent B (No) :", get_model_name(SKEPTIC_MODEL))
print("Adjudicator  :", get_model_name(ADJUDICATOR_MODEL))
print("Rounds       :", DEBATE_ROUNDS, "| Workers:", DEBATE_MAX_WORKERS)


In [ ]:
call_llm("hello world", model=EXTRACTOR_MODEL)

In [ ]:

json_url = "https://huggingface.co/datasets/theatticusproject/cuad/resolve/main/CUAD_v1/CUAD_v1.json"
json_path = "CUAD_v1.json"

df = download_parse_json(json_url, json_path)
df["clause_type"] = df["clause_name"].map(CLAUSE_TO_TYPE)

all_contracts = df["title"].unique()
# Change NUM_CONTRACTS in the config cell to process more contracts
selected_contracts = all_contracts[:NUM_CONTRACTS]

print(f"Loaded CUAD rows: {len(df):,}")
print(f"Contracts        : {len(all_contracts)}")
print(f"Selected         : {len(selected_contracts)}")
for idx, title in enumerate(selected_contracts, 1):
    contract_df = df[df["title"] == title]
    print(f"{idx}. {title[:80]}")
    print(f"   clauses={len(contract_df)} | gt_present={(~contract_df['is_impossible']).sum()}")


In [ ]:
_root = Path.cwd().parent if "Notebooks" in str(Path.cwd()) else Path.cwd()
pkl_dir = _root / "Notebooks" / "batch_results" / "extractions"
pkl_files = sorted(pkl_dir.glob("*.pkl"))

frames = []
for fp in pkl_files:
    with open(fp, "rb") as f:
        obj = pickle.load(f)

    if isinstance(obj, pd.DataFrame):
        part = obj
    elif isinstance(obj, dict) and "df" in obj and isinstance(obj["df"], pd.DataFrame):
        part = obj["df"]
    else:
        part = pd.DataFrame(obj)

    frames.append(part)

if not frames:
    raise FileNotFoundError(f"No .pkl files found in: {pkl_dir}")

df = pd.concat(frames, ignore_index=True)

# drop_duplicates needs hashable values; normalize nested list/dict columns in a temp frame
_tmp = df.copy()
for col in _tmp.columns:
    if _tmp[col].dtype == "object":
        _tmp[col] = _tmp[col].map(
            lambda x: json.dumps(x, sort_keys=True) if isinstance(x, (list, dict)) else x
        )

df = df.loc[~_tmp.duplicated()].reset_index(drop=True)

print(f"Loaded {len(pkl_files)} pkl files")
print(f"Unioned df shape: {df.shape}")
display(df.head())

In [ ]:
df_selected = df[(df['extraction_model'] == EXTRACTOR_MODEL) & (df['run_id'] == 1)]
print(f"Filtered to selected contracts: {len(df_selected)} rows")

In [ ]:
df_selected = df_selected.copy()
# Compute detection metrics
if df_selected is not None:
    # Confusion matrix calculation
    tp = ((df_selected['is_impossible'] == False) & (df_selected['is_impossible_ai'] == False)).sum()
    tn = ((df_selected['is_impossible'] == True) & (df_selected['is_impossible_ai'] == True)).sum()
    fp = ((df_selected['is_impossible'] == True) & (df_selected['is_impossible_ai'] == False)).sum()
    fn = ((df_selected['is_impossible'] == False) & (df_selected['is_impossible_ai'] == True)).sum()
    
    total_negative = (df_selected['is_impossible'] == True).sum()
    total_positive = (df_selected['is_impossible'] == False).sum()
    
    # Metrics calculation
    far = (fp / total_negative * 100) if total_negative > 0 else 0
    frr = (fn / total_positive * 100) if total_positive > 0 else 0
    detection_accuracy = ((tp + tn) / len(df_selected) * 100)
    
    print('='*80)
    print('📊 DETECTION METRICS')
    print('='*80)
    extraction_model = df_selected['extraction_model'].iloc[0] if 'extraction_model' in df_selected.columns else 'unknown'
    print(f'🤖 Extraction Model: {extraction_model}')
    print(f'\n🔍 Confusion Matrix:')
    print(f'   TP (Correct Detection):   {tp:4d}')
    print(f'   TN (Correct Abstention):  {tn:4d}')
    print(f'   FP (False Acceptance):    {fp:4d}')
    print(f'   FN (False Refusal):       {fn:4d}')
    print(f'\n📈 Metrics:')
    print(f'   FAR (False Acceptance):   {far:6.1f}%  (FP / GT-absent)')
    print(f'   FRR (False Refusal):      {frr:6.1f}%  (FN / GT-present)')
    print(f'   Detection Accuracy:       {detection_accuracy:6.1f}%  ((TP+TN) / Total)')
    print('='*80)
    
    # Store for later use
    benchmark_detection = {
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn),
        'FAR_%': round(far, 1),
        'FRR_%': round(frr, 1),
        'accuracy_%': round(detection_accuracy, 1)
    }
else:

    print('⚠️  No data available for detection metrics')    
    benchmark_detection = None

In [ ]:
_has_impossible_col = "is_impossible_ai" in df_selected.columns

contract_dict = {}
for title, group in df_selected.groupby("title", sort=False):
    clauses = []
    for _, row in group.iterrows():
        answer_ai = row["answer_ai"] if isinstance(row["answer_ai"], list) else []
        if _has_impossible_col:
            is_imp = bool(row["is_impossible_ai"])
        else:
            is_imp = not bool(answer_ai)
        clauses.append({
            "clause_name":      row["clause_name"],
            "answer_ai":        answer_ai,
            "is_impossible_ai": is_imp,
        })
    contract_dict[title] = {
        "context": group["context"].iloc[0],
        "clauses": clauses,
    }

# Quick check
first_title = list(contract_dict.keys())[0]
print(first_title[:80])
print(contract_dict[first_title]["clauses"][:2])
print(f"\nTotal contracts : {len(contract_dict)}")
print(f"Clauses (first) : {len(contract_dict[first_title]['clauses'])}")


In [ ]:
# =============================================================================
# State, Module-Level Prompts, and Helpers
# Prompts are defined here (outside functions) so they can be read,
# versioned, and tuned independently of the node logic.
# =============================================================================

# ── ClauseState ───────────────────────────────────────────────────────────────
class ClauseState(TypedDict):
    # Inputs
    contract_text:         str
    clause_name:           str
    clause_type:           str
    clause_definition:     str
    initial_answer:        list
    initial_is_impossible: bool
    # Debate bookkeeping
    round_num:      int
    max_rounds:     int
    debate_history: Annotated[list, operator.add]   # appended, not replaced
    # Per-round agent outputs (JSON strings)
    skeptic_argument:   str
    supporter_argument: str
    # Convergence
    consensus_reached: bool
    stability_score:   float
    # Final outputs
    final_answer:        list
    final_is_impossible: bool
    final_reasoning:     str
    # Re-extraction pass
    re_extracted_answer:     Optional[list]
    re_extraction_triggered: bool
    re_extraction_prompt:    Optional[str]
    pre_reextract_had_answer: Optional[bool]  # v8.6: True when reextract cleared non-empty initial
    verifier_result: Optional[str]             # v9.1: JSON string from verify_node for judge


# ── Shared policy blocks ──────────────────────────────────────────────────────
STRICT_CONTRACT_POLICY = (
    "You must reason ONLY from the provided contract text.\n"
    "Never import outside legal knowledge, prior beliefs, or template expectations.\n"
    "If evidence is not explicitly present in the contract text, do not assert it as fact.\n"
    "Prefer abstention (PARTIAL / UNSUPPORTED with null answers) over over-claiming."
)

SCENARIO_RUBRIC = (
    "Apply all relevant scenarios to the BASELINE extraction:\n"
    "1)  Explicit support: exact wording -> SUPPORTED.\n"
    "2)  Paraphrase: clear synonym with a quote -> PARTIAL unless unambiguous.\n"
    "3)  Partial: only subset supported -> PARTIAL.\n"
    "4)  Absent: no supporting quote anywhere -> UNSUPPORTED.\n"
    "5)  Over-claim: baseline adds details not in text -> UNSUPPORTED for those parts.\n"
    "6)  Conflict: two passages disagree -> PARTIAL, cite both.\n"
    "7)  Conditional: if/when/unless qualifier -> PARTIAL unless satisfied.\n"
    "8)  Exception/carve-out: must be reflected in answer.\n"
    "9)  Negation: shall vs shall not; may vs may not.\n"
    "10) Temporal (Expiration Date, Effective Date, Agreement Date):\n"
    "    A DESCRIPTION of term/condition (e.g. \'valid for 5 years\', \'term of 6 months\')\n"
    "    IS a valid extraction even without an exact calendar date. Mark SUPPORTED/PARTIAL,\n"
    "    not UNSUPPORTED, unless NO such description exists anywhere in the contract.\n"
    "11) Numeric: amounts/percentages must match text exactly.\n"
    "12) Party attribution: rights/obligations to correct party.\n"
    "13) Absence detection: if clause truly absent -> set refined_answer to null.\n"
    "\n"
    "SELF-CONSISTENCY RULE:\n"
    "  If you cite any baseline answer text verbatim in your evidence_quotes, you are\n"
    "  confirming that text EXISTS in the contract. You cannot simultaneously quote a\n"
    "  piece of text AND say it is UNSUPPORTED. Use PARTIAL instead.\n"
    "\n"
    "CRITICAL DEFINITION MATCH (absent->present flips):\n"
    "  Text must explicitly INSTANTIATE the clause definition -- NOT a loosely related concept.\n"
    "  Invalid matches:\n"
    "    - Recurring monthly fee != Minimum Commitment.\n"
    "      Minimum Commitment = a GUARANTEED MINIMUM PURCHASE QUANTITY/DOLLAR SPEND obligation.\n"
    "      Key test: Is the party required to purchase MORE than they might otherwise choose?\n"
    "      If no -> NOT Minimum Commitment.\n"
    "    - minimum quantities != Volume Restriction (= explicit upper cap on usage/sales).\n"
    "    - A fixed price mention != Price Restrictions.\n"
    "      Price Restrictions restrict CHANGES to price (cap/floor on future adjustments).\n"
    "      $X per hour is NOT a Price Restriction unless it restricts future price changes.\n"
    "  If uncertain whether found text fits the definition, use PARTIAL, not SUPPORTED.\n"
)

EVIDENCE_REQUIREMENTS = (
    "Evidence rules:\n"
    "  - Every material claim needs >=1 verbatim contract quote.\n"
    "  - Quotes must be exact text spans (not paraphrases).\n"
    "  - No quote for a claim -> mark that claim UNSUPPORTED.\n"
    "  - Confidence HIGH only when quotes are direct and unambiguous.\n"
    "\n"
    "VERDICT SEMANTICS:\n"
    "  SUPPORTED   = Extracted answer text is backed by contract text.\n"
    "  PARTIAL     = Some parts backed; others not.\n"
    "  UNSUPPORTED = Contract text does not support the extraction.\n"
    "\n"
    "  To vote UNSUPPORTED you MUST supply >=1 direct contradicting quote -- text that\n"
    "  explicitly negates, excludes, or makes the clause inapplicable.\n"
    "  A passage that does not mention the clause does NOT count as contradiction.\n"
    "  No such quote available -> vote PARTIAL, never UNSUPPORTED.\n"
    "\n"
    "  Confirming an EMPTY baseline is correct: verdict = SUPPORTED, refined_answer = null.\n"
    "  Do NOT fill refined_answer with an explanation of absence."
)

# ── Agent-specific context notes ──────────────────────────────────────────────
SKEPTIC_NOTE_EMPTY = (
    "NOTE -- BASELINE IS EMPTY:\n"
    "  To claim the clause IS present: (a) quote the exact contract text matching the\n"
    "  clause definition, (b) confirm it fits precisely (not loosely),\n"
    "  (c) if no such text exists, vote SUPPORTED to confirm absence is correct."
)

SKEPTIC_NOTE_NONEMPTY = (
    "NOTE -- BASELINE IS NON-EMPTY:\n"
    "  If you quote any part of the baseline answer in your evidence_quotes, you confirm\n"
    "  it exists in the contract. Use PARTIAL (not UNSUPPORTED) in that case.\n"
    "  DELETION RULE: To delete the baseline you MUST supply evidence_quotes containing\n"
    "  EXPLICIT NEGATION or CONTRADICTION -- text directly saying the clause does NOT apply,\n"
    "  is excluded, or is inapplicable. A passage that does not mention the clause is NOT\n"
    "  deletion evidence."
)

SUPPORTER_NOTE_EMPTY = (
    "NOTE -- BASELINE IS EMPTY:\n"
    "  To ADD a new answer: (a) quote the exact contract text, (b) confirm it directly\n"
    "  matches the clause definition (not loosely related), (c) set refined_answer to a\n"
    "  proper list of extracted answer strings (not null).\n"
    "  If confirming absence: verdict = SUPPORTED, refined_answer = null."
)

SUPPORTER_NOTE_NONEMPTY = (
    "NOTE -- BASELINE IS NON-EMPTY:\n"
    "  MANDATORY SEARCH (do this FIRST before forming any verdict):\n"
    "    Scan the ENTIRE contract for the exact baseline answer text. Read every section.\n"
    "    If >=8 consecutive words of the baseline appear verbatim in the contract: quote them.\n"
    "    THEN check: does the found text DIRECTLY INSTANTIATE the clause definition?\n"
    "    If text found BUT semantically does not match the definition (see CUAD DEFINITION\n"
    "    MISMATCH RULES) → vote PARTIAL or UNSUPPORTED based on definition fit, not mere presence.\n"
    "    Do not miss text in the middle of long contracts -- the relevant passage may be far from preamble.\n"
    "  To concede a point you MUST supply a contradicting contract quote.\n"
    "  Without a contradicting quote, preserve the baseline -- do not concede.\n"
    "  If you quote baseline answer text in your evidence_quotes, that confirms it\n"
    "  exists -> use PARTIAL (not UNSUPPORTED).\n"
    "  DELETION THRESHOLD: To vote UNSUPPORTED on a non-empty baseline you must supply\n"
    "  >=2 evidence quotes that each DIRECTLY contradict the baseline (explicit negation,\n"
    "  inapplicability, or textual exclusion -- NOT passages that are simply silent).\n"
    "  If you have only 1 contradicting quote -> vote PARTIAL, not UNSUPPORTED."
)

REEXTRACT_INSTRUCT = (
    "RE-EXTRACTION PROMPT -- HIGH-BAR ACTION FIELD (round 1 only):\n"
    "  Set re_extraction_prompt (non-null) ONLY when ALL of these hold:\n"
    "    1. You found a SPECIFIC passage DIRECTLY showing the extraction is objectively wrong.\n"
    "    2. The error is MATERIAL -- changes presence/absence or fundamentally misrepresents content.\n"
    "    3. You can name exactly WHERE the correct answer lives (verbatim quote >=10 words).\n"
    "  Leave null otherwise (normal uncertainty, round > 1, no direct contradicting quote, etc.).\n"
    "  Self-check: Do I have a passage PROVING this extraction is wrong? YES + material -> set."
)

REEXTRACT_ABSENT_BAR = (
    "IMPORTANT -- ORIGINAL BASELINE WAS EMPTY:\n"
    "  The prior extraction found NO instances of this clause.\n"
    "  To return a non-empty answer you must meet ALL three criteria:\n"
    "    1. >=2 DIRECT verbatim quotes that explicitly instantiate the clause definition.\n"
    "    2. HIGH CONFIDENCE the clause is present (not just plausible).\n"
    "    3. Quotes are unambiguous and directly name or enact the clause.\n"
    "  If you cannot meet all three, return answer = [] (preserve absence)."
)

VERIFY_PROMPT = (
    "You are a VERIFIER agent. Two debate agents concluded an extraction is absent/wrong.\n"
    "Your task: search the contract AND verify the definition fit of what you find.\n\n"
    "STEP 1 -- SEARCH (mandatory, read every section):\n"
    "  (a) Search for the EXACT extraction text.\n"
    "  (b) Partial match counts: >=8 consecutive words from the extraction in the contract = FOUND.\n"
    "  (c) Temporal clauses: a term description ('10-year term', 'valid for X years') IS a match.\n"
    "  (d) Report ALL matching passage(s) as evidence_quotes.\n\n"
    "STEP 2 -- DEFINITION FIT (apply AFTER finding text):\n"
    "  Even if text is found, check whether it DIRECTLY INSTANTIATES the clause DEFINITION.\n"
    "  Use the clause definition provided. If the found text negates, denies, or excludes\n"
    "  the clause type (instead of instantiating it), the definition does NOT fit.\n\n"
    "VERDICTS:\n"
    "  found=true,  verdict=SUPPORTED:   text found AND correctly instantiates the definition.\n"
    "  found=true,  verdict=PARTIAL:     text found but definition fit is uncertain/partial.\n"
    "  found=true,  verdict=UNSUPPORTED: text found BUT does NOT fit the clause definition.\n"
    "  found=false, verdict=UNSUPPORTED: no relevant text found anywhere.\n\n"
    "CRITICAL: Never hallucinate. Quote only text that literally appears in the contract."
)

# ── Helpers ───────────────────────────────────────────────────────────────────
def _llm_json(prompt: str, model: str, context: str = "") -> dict:
    """Call LLM and parse JSON. Always returns a dict (never None)."""
    try:
        raw = call_llm(prompt, model=model, temperature=0.0)
        result = parse_json_response(raw, context=context)
        return result if isinstance(result, dict) else {}
    except Exception as exc:
        print(f"  [{context}] LLM/JSON error: {exc}")
        return {}

_CLAUSE_DEF_MAP: dict = {
    item.partition(":")[0].strip().strip('"'):  item.partition(":")[2].strip()
    for item in CLAUSE_DEFINITIONS
    if isinstance(item, str) and ":" in item
}

def _get_clause_def(clause_name: str) -> str:
    return _CLAUSE_DEF_MAP.get(clause_name, f"A clause of type '{clause_name}'")

def _compute_stability(state) -> float:
    try:
        sk = json.loads(state.get("skeptic_argument", "{}")).get("verdict", "PARTIAL")
        su = json.loads(state.get("supporter_argument", "{}")).get("verdict", "PARTIAL")
    except Exception:
        return 0.5
    if sk == su and sk in ("SUPPORTED", "UNSUPPORTED"):
        return 1.0
    return 0.5 if "PARTIAL" in (sk, su) else 0.0

def _is_valid_answer_list(refined) -> bool:
    """True only if refined is a non-empty list of meaningful answer strings."""
    if not isinstance(refined, list) or not refined:
        return False
    absence_pfx = (
        "the contract does not", "no ", "not present", "not found",
        "absent", "this clause", "this agreement does not",
    )
    return any(
        isinstance(x, str) and x.strip()
        and not any(x.strip().lower().startswith(p) for p in absence_pfx)
        for x in refined
    )

print("State, prompts, and helpers defined.")


In [ ]:

# =============================================================================
# Graph Nodes — LangGraph Debate Workflow
#
# Clause-level graph:
#   START → skeptic → supporter → {finalize | loop-skeptic | arbiter | reextract}
#                                         ↓             ↓
#                                      (2+ rounds)  reextract → skeptic (R2)
#                                                   arbiter → finalize → END
# =============================================================================

def skeptic_node(state: ClauseState) -> dict:
    """Challenges the extraction with contract-only evidence."""
    round_n = state.get("round_num", 1)
    absent  = not bool(state.get("initial_answer"))
    note    = SKEPTIC_NOTE_EMPTY if absent else SKEPTIC_NOTE_NONEMPTY

    prev_section = ""
    if round_n > 1:
        try:
            prev_section = (
                f"\n\nSupporter's previous defense (round {round_n - 1}):\n"
                + json.loads(state.get("supporter_argument", "{}")).get("defense", "")
            )
        except Exception:
            pass

    contract_view = state["contract_text"]
    reext_block = REEXTRACT_INSTRUCT if round_n == 1 else ""

    prompt = "\n\n".join(filter(None, [
        "You are the SKEPTIC agent in a legal extraction debate.",
        STRICT_CONTRACT_POLICY,
        SCENARIO_RUBRIC,
        EVIDENCE_REQUIREMENTS,
        note,
        reext_block,
        f"Clause definition: {state['clause_definition']}",
        f"Contract text:\n{contract_view}",
        (f"Clause: {state['clause_name']} ({state['clause_type']})\n"
         f"BASELINE extraction:\n{state['initial_answer']}{prev_section}"),
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "challenge": "<specific critique referencing contract text>",\n'
         '  "evidence_quotes": ["<verbatim contract quotes>"],\n'
         '  "unsupported_spans": ["<parts of baseline not evidenced>"],\n'
         '  "scenario_flags": ["absent|partial_support|overclaim|conflict|conditional|exception|negation|temporal|numeric|party_mismatch"],\n'
         '  "suggested_correction": ["<corrected answer>"] or null,\n'
         '  "re_extraction_prompt": "<specific passage + what to fix>" or null,\n'
         '  "verdict": "SUPPORTED" | "PARTIAL" | "UNSUPPORTED",\n'
         '  "confidence": "HIGH" | "MEDIUM" | "LOW"\n'
         '}'),
    ]))

    result = _llm_json(prompt, model=SKEPTIC_MODEL, context=f"skeptic-r{round_n}")
    for k, v in [("verdict", "PARTIAL"), ("challenge", ""), ("evidence_quotes", []),
                 ("unsupported_spans", []), ("scenario_flags", []),
                 ("suggested_correction", None), ("re_extraction_prompt", None), ("confidence", "LOW")]:
        result.setdefault(k, v)

    return {
        "skeptic_argument": json.dumps(result),
        "debate_history": [{"round": round_n, "role": "skeptic", "argument": result}],
    }


def supporter_node(state: ClauseState) -> dict:
    """Defends the extraction with contract-only evidence; concedes only with contradicting quotes."""
    round_n = state.get("round_num", 1)
    absent  = not bool(state.get("initial_answer"))
    note    = SUPPORTER_NOTE_EMPTY if absent else SUPPORTER_NOTE_NONEMPTY

    try:
        sk_data = json.loads(state.get("skeptic_argument", "{}"))
    except Exception:
        sk_data = {}

    contract_view = state["contract_text"]

    prompt = "\n\n".join(filter(None, [
        "You are the SUPPORTER agent in a legal extraction debate.",
        STRICT_CONTRACT_POLICY,
        SCENARIO_RUBRIC,
        EVIDENCE_REQUIREMENTS,
        note,
        f"Clause definition: {state['clause_definition']}",
        f"Contract text:\n{contract_view}",
        (f"Clause: {state['clause_name']} ({state['clause_type']})\n"
         f"BASELINE extraction:\n{state['initial_answer']}\n\n"
         f"Skeptic challenge:\n{sk_data.get('challenge', '')}\n"
         f"Skeptic flags: {sk_data.get('scenario_flags', [])}\n"
         f"Skeptic quotes: {sk_data.get('evidence_quotes', [])}"),
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "defense": "<contract-grounded defense>",\n'
         '  "evidence_quotes": ["<verbatim supporting quotes>"],\n'
         '  "concessions": ["<points conceded — ONLY with contradicting quotes>"],\n'
         '  "scenario_flags": ["absent|partial_support|conflict|conditional|exception|negation|temporal|numeric|party_mismatch"],\n'
         '  "refined_answer": ["<precise answer strings from contract text>"] or null,\n'
         '  "verdict": "SUPPORTED" | "PARTIAL" | "UNSUPPORTED",\n'
         '  "confidence": "HIGH" | "MEDIUM" | "LOW"\n'
         '}'),
    ]))

    result = _llm_json(prompt, model=SUPPORTER_MODEL, context=f"supporter-r{round_n}")
    for k, v in [("verdict", "PARTIAL"), ("defense", ""), ("evidence_quotes", []),
                 ("concessions", []), ("scenario_flags", []),
                 ("refined_answer", None), ("confidence", "LOW")]:
        result.setdefault(k, v)
    if isinstance(result.get("refined_answer"), str):
        result["refined_answer"] = None

    return {
        "supporter_argument": json.dumps(result),
        "debate_history": [{"round": round_n, "role": "supporter", "argument": result}],
        "round_num": round_n + 1,
    }


def arbiter_node(state: ClauseState) -> dict:
    """Neutral arbiter resolves deadlock. Conservative default: preserve baseline unless strong contrary evidence."""
    history_text = "\n\n".join(
        f"--- Round {t['round']} | {t['role'].upper()} ---\n"
        f"Verdict: {t['argument'].get('verdict', '?')}\n"
        f"{'Challenge' if t['role'] == 'skeptic' else 'Defense'}: "
        f"{t['argument'].get('challenge' if t['role'] == 'skeptic' else 'defense', '')}\n"
        f"Flags: {t['argument'].get('scenario_flags', [])}\n"
        f"Evidence: {t['argument'].get('evidence_quotes', [])}"
        for t in state.get("debate_history", [])
    ) or "(no debate history)"

    contract_view = state["contract_text"]

    prompt = "\n\n".join([
        "You are the ARBITER in a legal extraction debate.",
        STRICT_CONTRACT_POLICY,
        SCENARIO_RUBRIC,
        EVIDENCE_REQUIREMENTS,
        ("Decision policy:\n"
         "  • Direct quote supports answer → keep/produce supported answer.\n"
         "  • Partial/conflicting/conditional → narrow answer or preserve baseline.\n"
         "  • No direct quote contradicting baseline → preserve baseline."),
        f"Clause definition: {state['clause_definition']}",
        f"Contract text:\n{contract_view}",
        (f"Clause: {state['clause_name']} ({state['clause_type']})\n"
         f"Original extraction: {state['initial_answer']}\n"
         f"Original impossible: {state['initial_is_impossible']}\n\n"
         f"Debate history:\n{history_text}"),
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "reasoning": "<contract-grounded rationale>",\n'
         '  "evidence_quotes": ["<decisive verbatim quotes>"],\n'
         '  "scenario_flags": ["absent|partial_support|conflict|conditional|exception|negation|temporal|numeric|party_mismatch"],\n'
         '  "final_answer": ["<strictly contract-grounded answer>"] or [],\n'
         '  "final_is_impossible": true | false,\n'
         '  "confidence": "HIGH" | "MEDIUM" | "LOW"\n'
         '}'),
    ])

    result = _llm_json(prompt, model=ADJUDICATOR_MODEL, context="arbiter")
    result.setdefault("reasoning", "Arbiter defaulted to original extraction.")
    result.setdefault("evidence_quotes", [])
    result.setdefault("scenario_flags", [])
    result.setdefault("final_answer", state.get("initial_answer", []))
    result["final_answer"] = result["final_answer"] or []
    result.setdefault("final_is_impossible", state.get("initial_is_impossible", False))
    result.setdefault("confidence", "LOW")

    return {
        "stability_score": _compute_stability(state),
        "consensus_reached": False,
        "final_answer": result["final_answer"],
        "final_is_impossible": result["final_is_impossible"],
        "final_reasoning": (
            f"[ARBITER] {result['reasoning']} | "
            f"flags={result['scenario_flags']} | "
            f"evidence={result['evidence_quotes']} (confidence: {result['confidence']})"
        ),
    }


def reextract_node(state: ClauseState) -> dict:
    """Second-pass re-extraction guided by Skeptic's error location."""
    try:
        sk_data = json.loads(state.get("skeptic_argument", "{}"))
    except Exception:
        sk_data = {}

    re_guidance = (
        sk_data.get("re_extraction_prompt")
        or sk_data.get("challenge", "Review the extraction for accuracy.")
    )
    sk_quotes = sk_data.get("evidence_quotes", [])
    original_was_absent = not bool(state.get("initial_answer"))
    contract_view = state["contract_text"]

    quote_section = (
        "\nRelevant passages identified by reviewer:\n"
        + "\n".join(f'  [{i+1}] "{q}"' for i, q in enumerate(sk_quotes[:5]))
        if sk_quotes else ""
    )
    absent_bar = ("\n\n" + REEXTRACT_ABSENT_BAR) if original_was_absent else ""

    prompt = "\n\n".join(filter(None, [
        ("You are a legal contract extraction specialist.\n"
         "A prior extraction has been reviewed and a specific material error was identified.\n"
         "Produce a corrected extraction addressing the reviewer's concerns."),
        STRICT_CONTRACT_POLICY,
        f"Clause: {state['clause_name']} ({state['clause_type']})\nClause definition: {state['clause_definition']}",
        f"Contract text:\n{contract_view}{quote_section}{absent_bar}",
        f"Prior extraction: {state['initial_answer']}\nReviewer critique: {re_guidance}",
        ("Instructions:\n"
         "  1. Address the specific error identified by the reviewer.\n"
         "  2. Ground every answer string in a verbatim contract quote.\n"
         "  3. If the clause is truly absent: answer = []."),
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "answer": ["<extracted answer strings>"] or [],\n'
         '  "is_impossible": true | false,\n'
         '  "evidence_quotes": ["<verbatim contract quotes>"],\n'
         '  "reasoning": "<what error was corrected and why>"\n'
         '}'),
    ]))

    result    = _llm_json(prompt, model=EXTRACTOR_MODEL, context="reextract")
    re_answer = result.get("answer", None)
    if not isinstance(re_answer, list):
        re_answer = state.get("initial_answer", [])
    prev_answer = state.get("initial_answer", [])

    print(f"  [reextract] '{state['clause_name']}' (orig_absent={original_was_absent}): {prev_answer} → {re_answer}")

    # If re-extractor cleared a previously non-empty answer → mark as confirmed absent.
    # This lets finalize correctly apply the strict absent-gate and set is_impossible=True.
    confirmed_absent = (re_answer == [] and not original_was_absent)
    result_dict = {
        "initial_answer":          re_answer,
        "re_extracted_answer":     re_answer,
        "re_extraction_triggered": True,
        "re_extraction_prompt":    re_guidance,
        "debate_history": [{
            "round": 0, "role": "reextractor",
            "argument": {
                "previous_answer":  prev_answer,
                "new_answer":       re_answer,
                "original_absent":  original_was_absent,
                "confirmed_absent": confirmed_absent,
                "reasoning":        result.get("reasoning", ""),
                "evidence_quotes":  result.get("evidence_quotes", []),
            },
        }],
    }
    if confirmed_absent:
        result_dict["initial_is_impossible"] = True
        # v8.6: track that original baseline had content (Case A) vs truly-absent (Case B)
        result_dict["pre_reextract_had_answer"] = True
        print(f"  [reextract] confirmed absent — setting initial_is_impossible=True")
    return result_dict


def verify_node(state: ClauseState) -> dict:
    """
    Independent verifier — searches contract for the baseline extraction and checks
    definition fit. Result stored in verifier_result for the judge to use.
    No deterministic overrides: the judge makes all final decisions.
    """
    initial_answer = state.get("initial_answer", [])
    # Use supporter's refined_answer if initial is empty (post-reextract scenario)
    verify_source = initial_answer
    if not verify_source:
        try:
            refined = json.loads(state.get("supporter_argument", "{}")).get("refined_answer") or []
            if refined:
                verify_source = refined
        except Exception:
            pass
    answer_sample = " | ".join(str(a)[:250] for a in verify_source[:2])

    prompt = "\n\n".join(filter(None, [
        "You are the VERIFIER agent in a contract analysis pipeline.",
        VERIFY_PROMPT,
        STRICT_CONTRACT_POLICY,
        f"Clause: {state['clause_name']} ({state['clause_type']})\nClause definition: {state['clause_definition']}",
        f"Contract text:\n{state['contract_text']}",
        f"Extraction to verify:\n{answer_sample}",
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "found": true | false,\n'
         '  "evidence_quotes": ["<verbatim contract quotes>"] or [],\n'
         '  "verdict": "SUPPORTED" | "PARTIAL" | "UNSUPPORTED",\n'
         '  "reasoning": "<explain what you found or did not find>"\n'
         '}'),
    ]))

    result  = _llm_json(prompt, model=ADJUDICATOR_MODEL, context="verify")
    found   = result.get("found", False)
    vq      = result.get("evidence_quotes", []) or []
    verdict = result.get("verdict", "UNSUPPORTED")

    print(f"  [verify] '{state['clause_name']}': found={found} verdict={verdict} nq={len(vq)}")

    return {
        "verifier_result": json.dumps({
            "found": found,
            "verdict": verdict,
            "evidence_quotes": vq,
            "reasoning": result.get("reasoning", ""),
        }),
        "debate_history": [{"round": 0, "role": "verifier", "argument": {
            "found": found, "verdict": verdict,
            "evidence_quotes": vq, "reasoning": result.get("reasoning", ""),
        }}],
    }
def judge_node(state: ClauseState) -> dict:
    """
    v9.2 LLM Judge with verifier-gated absent→present.
    
    The judge makes ALL content-level decisions via LLM reasoning.
    Structural guard: for absent→present additions, the judge's decision
    is only accepted if the independent verifier also confirmed the clause
    (found=true AND verdict in SUPPORTED/PARTIAL). This prevents the judge
    from hallucinating additions when no agent found evidence.
    """
    history = state.get("debate_history", []) or []
    history_text = "\n\n".join(
        f"--- {t.get('role','?').upper()} (round {t.get('round','?')}) ---\n"
        + "\n".join(f"  {k}: {v}" for k, v in (t.get("argument") or {}).items())
        for t in history
    ) or "(no debate history)"

    # ── Build structured debate consensus summary ──
    try:
        sk_data = json.loads(state.get("skeptic_argument", "{}"))
        su_data = json.loads(state.get("supporter_argument", "{}"))
    except Exception:
        sk_data = su_data = {}

    sk_verdict = sk_data.get("verdict", "PARTIAL")
    su_verdict = su_data.get("verdict", "PARTIAL")
    sk_quotes  = sk_data.get("evidence_quotes", []) or []
    su_quotes  = su_data.get("evidence_quotes", []) or []
    sk_flags   = sk_data.get("scenario_flags", []) or []
    su_flags   = su_data.get("scenario_flags", []) or []

    consensus_summary = (
        f"DEBATE CONSENSUS SUMMARY:\n"
        f"  Skeptic verdict: {sk_verdict} (confidence: {sk_data.get('confidence','?')}, quotes: {len(sk_quotes)}, flags: {sk_flags})\n"
        f"  Supporter verdict: {su_verdict} (confidence: {su_data.get('confidence','?')}, quotes: {len(su_quotes)}, flags: {su_flags})\n"
        f"  Baseline was absent: {state.get('initial_is_impossible', False)}\n"
        f"  Total supporting quotes from debate: {len(sk_quotes) + len(su_quotes)}"
    )

    verifier_summary = ""
    verifier_confirmed = False  # Used for absent→present gate
    if state.get("verifier_result"):
        try:
            vr = json.loads(state["verifier_result"])
            vq_text = "\n".join(f'  \"{q}\"' for q in (vr.get("evidence_quotes") or [])[:3])
            verifier_confirmed = (vr.get("found") == True and vr.get("verdict") == "SUPPORTED")
            verifier_summary = (
                f"\n\nINDEPENDENT VERIFIER ASSESSMENT:\n"
                f"  found_in_contract: {vr.get('found')}\n"
                f"  definition_fit_verdict: {vr.get('verdict')}\n"
                f"  reasoning: {vr.get('reasoning','')}\n"
                f"  supporting_quotes:\n{vq_text or '  (none)'}"
            )
        except Exception:
            pass

    prompt = "\n\n".join([
        "You are the JUDGE in a multi-agent legal extraction debate. Make the FINAL binding decision.",
        STRICT_CONTRACT_POLICY,
        SCENARIO_RUBRIC,
        EVIDENCE_REQUIREMENTS,
        VERIFY_PROMPT,
        (f"CLAUSE: {state['clause_name']} ({state['clause_type']})\n"
         f"DEFINITION: {state['clause_definition']}"),
        (f"ORIGINAL BASELINE EXTRACTION:\n"
         f"  answer: {state['initial_answer']}\n"
         f"  marked_absent: {state.get('initial_is_impossible', False)}"),
        f"CONTRACT TEXT:\n{state['contract_text']}",
        consensus_summary,
        f"FULL DEBATE TRANSCRIPT:\n{history_text}{verifier_summary}",
        ("FINAL DECISION INSTRUCTIONS:\n"
         "1. Weigh ALL debate evidence and the verifier assessment against the actual contract text.\n"
         "2. Apply the clause DEFINITION strictly — read the NOTE section carefully if present.\n"
         "3. Apply all definition-fit rules from VERIFY_PROMPT strictly.\n"
         "4. To confirm a PRESENT answer: require at least one verbatim quote that DIRECTLY instantiates the definition.\n"
         "5. To DELETE a present baseline (present→absent): the evidence must show the text\n"
         "   does NOT match the clause definition (negation, denial, or definition mismatch).\n"
         "6. To ADD a new answer (absent→present): require STRONG, DIRECT, UNAMBIGUOUS evidence.\n"
         "7. When uncertain, preserve the original baseline (do not add or delete).\n"
         "8. If final_answer is an empty list, set final_is_impossible=true."),
        ('Return ONLY valid JSON:\n'
         '{\n'
         '  "reasoning": "<concise contract-grounded rationale>",\n'
         '  "evidence_quotes": ["<decisive verbatim quotes>"],\n'
         '  "final_answer": ["<answer strings>"] or [],\n'
         '  "final_is_impossible": true | false,\n'
         '  "confidence": "HIGH" | "MEDIUM" | "LOW"\n'
         '}'),
    ])

    result = _llm_json(prompt, model=ADJUDICATOR_MODEL, context="judge")
    result.setdefault("final_answer", state.get("initial_answer", []))
    result["final_answer"] = result["final_answer"] or []
    result.setdefault("final_is_impossible", state.get("initial_is_impossible", False))
    result.setdefault("reasoning", "Judge defaulted to original extraction.")
    result.setdefault("evidence_quotes", [])
    result.setdefault("confidence", "LOW")

    # Logical consistency: empty answer = absent
    if not result["final_answer"]:
        result["final_is_impossible"] = True

    # ── v9.2 Structural guard: verifier-gated absent→present ──
    # If baseline was absent and judge wants to add an answer,
    # only allow it if the independent verifier confirmed the clause exists.
    _orig_had_answer = bool(state.get("pre_reextract_had_answer", False))
    baseline_was_absent = bool(state.get("initial_is_impossible", False)) and not _orig_had_answer
    # ── DEBATE CONSENSUS CHECK for absent→present ──
    debate_consensus_supported = False
    for h in reversed(state.get("debate_history", [])):
        if h.get("role") == "skeptic":
            sk_v = (h.get("argument") or {}).get("verdict", "")
            break
    else:
        sk_v = ""
    for h in reversed(state.get("debate_history", [])):
        if h.get("role") == "supporter":
            su_v = (h.get("argument") or {}).get("verdict", "")
            break
    else:
        su_v = ""
    debate_consensus_supported = (sk_v == "SUPPORTED" and su_v == "SUPPORTED")

    judge_wants_to_add = baseline_was_absent and bool(result["final_answer"]) and not result["final_is_impossible"]
    if judge_wants_to_add and (not verifier_confirmed or not debate_consensus_supported):
        print(f"  [judge] VERIFIER GATE blocked absent→present for '{state['clause_name']}': verifier did not confirm clause")
        result["final_answer"] = []
        result["final_is_impossible"] = True
        result["reasoning"] = f"[VERIFIER GATE] Judge wanted to add but verifier did not confirm clause exists. {result['reasoning']}"

    # ── DELETION GATE: block present→absent if verifier says clause IS present ──
    baseline_was_present = not baseline_was_absent
    judge_wants_to_delete = baseline_was_present and (not result["final_answer"] or result["final_is_impossible"])
    verifier_says_present = False
    if state.get("verifier_result"):
        try:
            vr_del = json.loads(state["verifier_result"])
            verifier_says_present = (vr_del.get("found") == True and vr_del.get("verdict") in ("SUPPORTED", "PARTIAL"))
        except Exception:
            pass
    if judge_wants_to_delete and verifier_says_present:
        print(f"  [judge] DELETION GATE blocked present→absent for '{state['clause_name']}': verifier confirmed clause exists")
        result["final_answer"] = state.get("initial_answer", [])
        result["final_is_impossible"] = False
        result["reasoning"] = f"[DELETION GATE] Debate wanted to remove but verifier confirmed clause exists in contract. {result['reasoning']}"

    rounds_done = state.get("round_num", 1) - 1
    _fa = result['final_answer'] or []
    print(f"  [judge] '{state['clause_name']}': answer={_fa[:1]}{'...' if len(_fa) > 1 else ''} impossible={result['final_is_impossible']} conf={result['confidence']}")
    return {
        "stability_score":   _compute_stability(state),
        "consensus_reached": True,
        "final_answer":        result["final_answer"],
        "final_is_impossible": result["final_is_impossible"],
        "final_reasoning": (
            f"[JUDGE after {rounds_done} round(s)] {result['reasoning']}\n"
            f"evidence={result['evidence_quotes']} (confidence: {result['confidence']})"
        ),
    }


def route_after_supporter(
    state: ClauseState,
) -> Literal["skeptic_node", "arbiter_node", "verify_node", "reextract_node"]:
    """
    v9.1: Simplified agentic routing.
    All consensus → verify → judge.  No content-based if/else logic.
    Round 1 only: if skeptic flagged a material error → reextract (fires once).
    """
    try:
        sk_data    = json.loads(state.get("skeptic_argument", "{}"))
        sk_verdict = sk_data.get("verdict", "PARTIAL")
        su_verdict = json.loads(state.get("supporter_argument", "{}")).get("verdict", "PARTIAL")
        sk_reext   = sk_data.get("re_extraction_prompt")
    except Exception:
        sk_verdict = su_verdict = "PARTIAL"
        sk_reext   = None

    rounds_completed    = state.get("round_num", 1) - 1
    max_rounds          = state.get("max_rounds", DEBATE_ROUNDS)
    already_reextracted = bool(state.get("re_extraction_triggered", False))

    # R1 only: skeptic identifies a material extraction error → re-extract before continuing
    if rounds_completed == 1 and not already_reextracted and bool(sk_reext):
        return "reextract_node"

    # Consensus (both agree on SUPPORTED or UNSUPPORTED) → verify then judge
    verdicts_agree = (sk_verdict == su_verdict) and (sk_verdict in ("SUPPORTED", "UNSUPPORTED"))
    if verdicts_agree:
        return "verify_node"

    # More rounds available → continue debate
    if rounds_completed < max_rounds:
        return "skeptic_node"

    # Rounds exhausted without consensus → arbiter → verify → judge
    return "arbiter_node"

print("Graph nodes defined (v9.5: balanced gates (add+delete) + softened defs + agentic judge).")


In [ ]:

# ── Build clause-level debate graph ──────────────────────────────────────────
clause_debate_builder = StateGraph(ClauseState)

# Register nodes
clause_debate_builder.add_node("skeptic_node",   skeptic_node)
clause_debate_builder.add_node("supporter_node", supporter_node)
clause_debate_builder.add_node("arbiter_node",   arbiter_node)
clause_debate_builder.add_node("judge_node",     judge_node)
clause_debate_builder.add_node("reextract_node", reextract_node)   # v7: feedback-to-extractor
clause_debate_builder.add_node("verify_node",    verify_node)       # v8: deletion verifier

# Edges
clause_debate_builder.add_edge(START,           "skeptic_node")
clause_debate_builder.add_edge("skeptic_node",  "supporter_node")

# Conditional routing after every supporter response
clause_debate_builder.add_conditional_edges(
    "supporter_node",
    route_after_supporter,
    {
        "skeptic_node":   "skeptic_node",    # loop back (R2+)
        "arbiter_node":   "arbiter_node",    # fallback judge
        "reextract_node": "reextract_node",  # v7: re-extraction after R1
        "verify_node":    "verify_node",      # consensus or UNSUPPORTED → verify → judge
    },
)

# After re-extraction → restart debate with corrected answer
clause_debate_builder.add_edge("reextract_node", "skeptic_node")

# After verification → judge (judge makes final decision using verifier evidence)
clause_debate_builder.add_edge("verify_node",    "judge_node")

# v9.1: arbiter → verify (collect evidence) → judge (final decision)
clause_debate_builder.add_edge("arbiter_node",  "verify_node")
clause_debate_builder.add_edge("judge_node", END)

# ── Compile with InMemorySaver checkpointer ───────────────────────────────────
# Each debate session gets its own thread_id; state is snapshotted after every node.
# Use get_session_state(thread_id) / get_session_history(thread_id) to inspect.
_debate_checkpointer = InMemorySaver()
clause_debate_graph = clause_debate_builder.compile(checkpointer=_debate_checkpointer)

print("Clause debate graph compiled with InMemorySaver checkpointer.")
print("Nodes: skeptic → supporter → {finalize | verify | loop | arbiter | reextract} → ...")
print()

# Optional: visualise the graph structure (requires graphviz/mermaid)
try:
    from IPython.display import Image, display
    display(Image(clause_debate_graph.get_graph().draw_mermaid_png()))
except Exception:
    print(clause_debate_graph.get_graph().draw_mermaid())


In [ ]:

import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

_print_lock = threading.Lock()


# ── Filename and DataFrame helpers ────────────────────────────────────────────
def _safe_filename(title: str) -> str:
    """Convert a contract title to a safe filesystem stem (max 120 chars)."""
    return re.sub(r'[^\w\-]', '_', title)[:120]


def _build_debate_df(debate_results: dict) -> pd.DataFrame:
    """Flatten all_debate_results into a DataFrame with one row per clause."""
    rows = []
    for title, clause_list in debate_results.items():
        for r in clause_list:
            rows.append({
                "title":                title,
                "clause_name":          r["clause_name"],
                "debate_answer":        r["final_answer"],
                "debate_is_impossible": r["final_is_impossible"],
                "debate_consensus":     r["consensus_reached"],
                "debate_stability":     r["stability_score"],
                "debate_rounds":        r["rounds_used"],
                "debate_reasoning":     r["final_reasoning"],
                "debate_changed":       (r["initial_answer"] != r["final_answer"])
                                        or (r["initial_is_impossible"] != r["final_is_impossible"]),
            })
    return pd.DataFrame(rows)


# ── Single-clause debate runner ───────────────────────────────────────────────
def run_clause_debate(
    contract_text: str,
    clause_name: str,
    initial_answer: list,
    initial_is_impossible: bool,
    max_rounds: int = DEBATE_ROUNDS,
    thread_id: str = None,
) -> dict:
    """
    Run the LangGraph debate graph for one clause.
    Returns a dict with: clause_name, final_answer, final_is_impossible,
    final_reasoning, consensus_reached, stability_score, rounds_used,
    debate_history, thread_id, re_extraction_triggered, re_extracted_answer.
    """
    clause_type       = CLAUSE_TO_TYPE.get(clause_name, "general")
    clause_definition = _get_clause_def(clause_name)
    thread_id         = thread_id or f"{clause_name.replace(' ', '_')}-{uuid.uuid4().hex[:8]}"
    _config           = {"configurable": {"thread_id": thread_id}}

    initial_state: ClauseState = {
        "contract_text":         contract_text,
        "clause_name":           clause_name,
        "clause_type":           clause_type,
        "clause_definition":     clause_definition,
        "initial_answer":        initial_answer,
        "initial_is_impossible": initial_is_impossible,
        "round_num":             1,
        "max_rounds":            max_rounds,
        "debate_history":        [],
        "skeptic_argument":      "",
        "supporter_argument":    "",
        "consensus_reached":     False,
        "stability_score":       0.0,
        "final_answer":          [],
        "final_is_impossible":   initial_is_impossible,
        "final_reasoning":       "",
        "re_extracted_answer":      None,
        "re_extraction_triggered":  False,
        "re_extraction_prompt":     None,
    }

    result = clause_debate_graph.invoke(initial_state, _config)
    return {
        "clause_name":             clause_name,
        "initial_answer":          initial_answer,
        "initial_is_impossible":   initial_is_impossible,
        "final_answer":            result.get("final_answer", initial_answer),
        "final_is_impossible":     result.get("final_is_impossible", initial_is_impossible),
        "final_reasoning":         result.get("final_reasoning", ""),
        "consensus_reached":       result.get("consensus_reached", False),
        "stability_score":         result.get("stability_score", 0.0),
        "rounds_used":             result.get("round_num", 1) - 1,
        "debate_history":          result.get("debate_history", []),
        "thread_id":               thread_id,
        "re_extraction_triggered": result.get("re_extraction_triggered", False),
        "re_extracted_answer":     result.get("re_extracted_answer", None),
    }


# ── Contract-level debate runner (parallel) ───────────────────────────────────
def run_contract_debate(
    contract_title: str,
    contract_entry: dict,
    max_rounds: int = DEBATE_ROUNDS,
    skip_impossible: bool = False,
    verbose: bool = True,
    max_workers: int = DEBATE_MAX_WORKERS,
) -> list:
    """
    Run the debate workflow over all clauses in one contract in parallel.
    Returns a list of per-clause result dicts in original clause order.
    """
    contract_text = contract_entry.get("context", "")
    clauses       = contract_entry.get("clauses", [])
    total         = len(clauses)

    if verbose:
        print(f"\n{'='*70}\nContract: {contract_title[:80]}\nClauses : {total}  (workers={max_workers})\n{'='*70}")

    def _debate_one(idx: int, clause: dict):
        clause_name      = clause.get("clause_name", "")
        answer_ai        = clause.get("answer_ai", [])
        is_impossible_ai = clause.get("is_impossible_ai", not bool(answer_ai))
        thread_id = (
            f"{contract_title[:30].replace(' ', '_')}"
            f"-{clause_name.replace(' ', '_')}"
            f"-{uuid.uuid4().hex[:6]}"
        )

        if skip_impossible and is_impossible_ai:
            res = {
                "clause_name": clause_name, "initial_answer": answer_ai,
                "initial_is_impossible": is_impossible_ai, "final_answer": answer_ai,
                "final_is_impossible": is_impossible_ai,
                "final_reasoning": "[SKIPPED — initial extraction marked impossible]",
                "consensus_reached": True, "stability_score": 1.0, "rounds_used": 0,
                "debate_history": [], "thread_id": thread_id,
                "re_extraction_triggered": False, "re_extracted_answer": None,
            }
            if verbose:
                with _print_lock:
                    print(f"  [{idx:2d}/{total}] {clause_name[:55]:<55} [skipped]")
            return idx, res

        t0 = time.time()
        try:
            res = run_clause_debate(
                contract_text=contract_text, clause_name=clause_name,
                initial_answer=answer_ai, initial_is_impossible=is_impossible_ai,
                max_rounds=max_rounds, thread_id=thread_id,
            )
        except Exception as exc:
            traceback.print_exc()
            res = {
                "clause_name": clause_name, "initial_answer": answer_ai,
                "initial_is_impossible": is_impossible_ai, "final_answer": answer_ai,
                "final_is_impossible": is_impossible_ai,
                "final_reasoning": f"[ERROR] {exc}",
                "consensus_reached": False, "stability_score": 0.0, "rounds_used": 0,
                "debate_history": [], "thread_id": thread_id,
                "re_extraction_triggered": False, "re_extracted_answer": None,
            }

        if verbose:
            reext_flag = " ↻reextract" if res.get("re_extraction_triggered") else ""
            status     = "✓ consensus" if res["consensus_reached"] else "✗ arbiter"
            with _print_lock:
                print(
                    f"  [{idx:2d}/{total}] {clause_name[:50]:<50} answers={len(answer_ai)}"
                    f"  {status}  rounds={res['rounds_used']}"
                    f"  stab={res['stability_score']:.1f}"
                    f"  {reext_flag}  [{time.time()-t0:.1f}s]"
                )
        return idx, res

    results_map: dict = {}
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(_debate_one, idx, clause): idx
                   for idx, clause in enumerate(clauses, 1)}
        for future in as_completed(futures):
            idx, res = future.result()
            results_map[idx] = res

    return [results_map[i] for i in range(1, total + 1)]


# ── Session inspection utilities ──────────────────────────────────────────────
def get_session_state(thread_id: str):
    """Return the latest checkpointed state snapshot for a debate session."""
    return clause_debate_graph.get_state({"configurable": {"thread_id": thread_id}})

def get_session_history(thread_id: str) -> list:
    """Return the full checkpoint history for a debate session (newest first)."""
    return list(clause_debate_graph.get_state_history({"configurable": {"thread_id": thread_id}}))

def list_active_sessions() -> list:
    """Return all thread_ids currently stored in the checkpointer."""
    return [c.config["configurable"]["thread_id"] for c in _debate_checkpointer.list(None)]


print("run_clause_debate(), run_contract_debate(), session utils ready.")
print(f"Parallel workers: {DEBATE_MAX_WORKERS}  (set DEBATE_MAX_WORKERS=1 to disable)")


In [ ]:

# ── Multi-contract debate batch loop ──────────────────────────────────────────
# Per-contract: saves <EXTRACTION_DIR>/<title>.pkl immediately after each run.
# Every EXTRACTION_CHECKPOINT_INTERVAL contracts: saves a cumulative merged snapshot.
# Resume: contracts whose .pkl already exists are skipped automatically.

VERBOSE_TRANSCRIPT = False   # set True to print full per-clause debate turns

all_debate_results: dict = {}
_total_contracts = len(contract_dict)
_batch_t0        = time.time()
_contracts_done  = 0

for _ci, (_title, _entry) in enumerate(contract_dict.items(), 1):
    _safe = _safe_filename(_title)
    _ckpt = EXTRACTION_DIR / f"{_safe}.pkl"

    if _ckpt.exists():
        with open(_ckpt, "rb") as _fh:
            all_debate_results[_title] = pickle.load(_fh)
        print(f"[{_ci:2d}/{_total_contracts}] LOADED  {_title[:70]}")
        _contracts_done += 1
    else:
        print(f"[{_ci:2d}/{_total_contracts}] RUNNING {_title[:70]}")
        _ct0     = time.time()
        _results = run_contract_debate(
            contract_title=_title, contract_entry=_entry,
            max_rounds=DEBATE_ROUNDS, skip_impossible=False, verbose=True,
        )
        _elapsed = time.time() - _ct0

        with open(_ckpt, "wb") as _fh:
            pickle.dump(_results, _fh)
        with open(DEBATE_LOG_DIR / f"{_safe}.json", "w") as _fh:
            json.dump(_results, _fh, indent=2, default=str)

        all_debate_results[_title] = _results
        _contracts_done += 1

        _nc = sum(1 for r in _results if r["consensus_reached"])
        _ch = sum(1 for r in _results if r["initial_answer"] != r["final_answer"]
                  or r["initial_is_impossible"] != r["final_is_impossible"])
        print(f"    → {len(_results)} clauses | consensus={_nc} | changed={_ch} | {_elapsed:.1f}s")

        if VERBOSE_TRANSCRIPT:
            for res in _results:
                print(f"\n{'─'*72}\nCLAUSE : {res['clause_name']}")
                print(f"Initial: {res['initial_answer']}  impossible={res['initial_is_impossible']}")
                print(f"Final  : {res['final_answer']}  impossible={res['final_is_impossible']}")
                for turn in res.get("debate_history", []):
                    role = turn["role"].upper()
                    arg  = turn["argument"]
                    key  = "challenge" if role == "SKEPTIC" else "defense"
                    print(f"\n  [{role} R{turn['round']}] verdict={arg.get('verdict','?')}")
                    print(f"  {arg.get(key,'')[:200]}")

    if _contracts_done % EXTRACTION_CHECKPOINT_INTERVAL == 0:
        _snap_path = CHECKPOINT_DIR / f"batch_checkpoint_{_contracts_done:04d}.pkl"
        _build_debate_df(all_debate_results).to_pickle(_snap_path)
        print(f"\n  ── Interval checkpoint @ {_contracts_done} contracts → {_snap_path.name}\n")

_total_elapsed = time.time() - _batch_t0
_total_clauses = sum(len(v) for v in all_debate_results.values())
print(f"\n{'='*60}\nBATCH COMPLETE")
print(f"  Contracts : {len(all_debate_results)} / {_total_contracts}")
print(f"  Clauses   : {_total_clauses}  |  Time: {_total_elapsed:.1f}s")
print(f"  PKLs      : {EXTRACTION_DIR}")
print(f"{'='*60}")


In [ ]:
# Reset debate outputs to force rerun with latest prompt policy
for p in list(EXTRACTION_DIR.glob("*.pkl")) + list(DEBATE_LOG_DIR.glob("*.json")):
    p.unlink(missing_ok=True)
for p in CHECKPOINT_DIR.glob("batch_checkpoint_*.pkl"):
    p.unlink(missing_ok=True)
print("Cleared prior debate artifacts from extractions/, debate_logs/, checkpoints/")

In [ ]:

# ── (Optional) Reload all_debate_results from disk ────────────────────────────
# Run this cell standalone to rebuild all_debate_results from saved checkpoints
# without rerunning the debate (e.g. after a kernel restart).

_loaded: dict = {}
for _pkl in sorted(EXTRACTION_DIR.glob("*.pkl")):
    with open(_pkl, "rb") as _fh:
        _results = pickle.load(_fh)
    # Recover the contract title from the first result record
    if _results and isinstance(_results, list) and "clause_name" in _results[0]:
        # Match back to contract_dict if available; else use filename stem
        _matched = [t for t in contract_dict if _safe_filename(t) == _pkl.stem]
        _key = _matched[0] if _matched else _pkl.stem
        _loaded[_key] = _results

if _loaded:
    all_debate_results = _loaded
    print(f"Reloaded {len(all_debate_results)} contracts from {EXTRACTION_DIR}")
    for _t, _r in all_debate_results.items():
        print(f"  {_t[:70]}  ({len(_r)} clauses)")
else:
    print("No checkpoint files found — all_debate_results comes from the debate run above.")


In [ ]:

# ── Build results DataFrame for comparison ────────────────────────────────────
records = []
for title, clause_results in all_debate_results.items():
    for r in clause_results:
        records.append({
            "contract":              title,
            "clause_name":           r["clause_name"],
            "initial_impossible":    r["initial_is_impossible"],
            "final_impossible":      r["final_is_impossible"],
            "impossible_changed":    r["initial_is_impossible"] != r["final_is_impossible"],
            "initial_n_answers":     len(r["initial_answer"]),
            "final_n_answers":       len(r["final_answer"]),
            "answers_changed":       r["initial_answer"] != r["final_answer"],
            "consensus_reached":     r["consensus_reached"],
            "stability_score":       r["stability_score"],
            "rounds_used":           r["rounds_used"],
            "re_extraction_triggered": r.get("re_extraction_triggered", False),
        })

df_debate = pd.DataFrame(records)

n_reextracted = df_debate["re_extraction_triggered"].sum()

print("Debate results summary (v7 — feedback-to-extractor)")
print("=" * 60)
print(f"Total clauses        : {len(df_debate)}")
print(f"Consensus reached    : {df_debate['consensus_reached'].sum()} "
      f"({df_debate['consensus_reached'].mean()*100:.1f}%)")
print(f"Avg stability        : {df_debate['stability_score'].mean():.2f}")
print(f"Avg rounds used      : {df_debate['rounds_used'].mean():.2f}")
print(f"Impossible changed   : {df_debate['impossible_changed'].sum()}")
print(f"Answers changed      : {df_debate['answers_changed'].sum()}")
print(f"Re-extraction fired  : {n_reextracted} ({n_reextracted/len(df_debate)*100:.1f}%)")
print()

# Breakdown by consensus
print(df_debate.groupby("consensus_reached")[["stability_score", "rounds_used"]].mean().round(2))
print()

# Re-extraction breakdown: how often did re-extraction also change the final answer?
if n_reextracted > 0:
    reext_df = df_debate[df_debate["re_extraction_triggered"]]
    reext_changed = (reext_df["answers_changed"] | reext_df["impossible_changed"]).sum()
    print(f"Of {n_reextracted} re-extractions: {reext_changed} also changed final answer "
          f"({reext_changed/n_reextracted*100:.1f}%)")
    print()

# Show clauses where answers changed
changed = df_debate[df_debate["answers_changed"] | df_debate["impossible_changed"]]
if not changed.empty:
    print(f"\nClauses where debate changed the extraction ({len(changed)}):")
    display_cols = ["clause_name", "initial_impossible", "final_impossible",
                    "initial_n_answers", "final_n_answers", "rounds_used",
                    "stability_score", "re_extraction_triggered"]
    display(changed[display_cols].reset_index(drop=True))
else:
    print("No extractions were changed by the debate.")


In [ ]:

# ── Debate vs Single-Pass Comparison ─────────────────────────────────────────
# Merge ground-truth labels from df_selected into df_debate
gt = (df_selected[["title", "clause_name", "is_impossible"]]
      .drop_duplicates(["title", "clause_name"])
      .rename(columns={"title": "contract"}))
df_cmp = df_debate.merge(gt, on=["contract", "clause_name"], how="inner")

# ── Confusion matrices ────────────────────────────────────────────────────────
def confusion_metrics(pred_impossible, gt_impossible):
    tp = ((gt_impossible == False) & (pred_impossible == False)).sum()
    tn = ((gt_impossible == True)  & (pred_impossible == True)).sum()
    fp = ((gt_impossible == True)  & (pred_impossible == False)).sum()
    fn = ((gt_impossible == False) & (pred_impossible == True)).sum()
    total_neg = (gt_impossible == True).sum()
    total_pos = (gt_impossible == False).sum()
    far = fp / total_neg * 100 if total_neg else 0
    frr = fn / total_pos * 100 if total_pos else 0
    acc = (tp + tn) / len(gt_impossible) * 100 if len(gt_impossible) else 0
    return dict(TP=int(tp), TN=int(tn), FP=int(fp), FN=int(fn),
                FAR=round(far, 1), FRR=round(frr, 1), ACC=round(acc, 1))

baseline_m = confusion_metrics(df_cmp["initial_impossible"], df_cmp["is_impossible"])
debate_m   = confusion_metrics(df_cmp["final_impossible"],   df_cmp["is_impossible"])

# ── Print comparison table ────────────────────────────────────────────────────
print("=" * 60)
print("  SINGLE-PASS vs DEBATE — Detection Metric Comparison")
print("=" * 60)
header = f"{'Metric':<22} {'Single-Pass':>12} {'Debate':>12} {'Delta':>10}"
print(header)
print("-" * 60)
for key in ["TP", "TN", "FP", "FN"]:
    delta = debate_m[key] - baseline_m[key]
    sign  = "+" if delta > 0 else ""
    print(f"  {key:<20} {baseline_m[key]:>12} {debate_m[key]:>12} {sign}{delta:>9}")
print("-" * 60)
for key, label in [("FAR", "FAR % (↓ better)"), ("FRR", "FRR % (↓ better)"), ("ACC", "Accuracy % (↑ better)")]:
    delta = debate_m[key] - baseline_m[key]
    sign  = "+" if delta > 0 else ""
    print(f"  {label:<20} {baseline_m[key]:>11.1f}% {debate_m[key]:>11.1f}% {sign}{delta:>8.1f}%")
print("=" * 60)

# ── Summary verdict ───────────────────────────────────────────────────────────
acc_delta = debate_m["ACC"] - baseline_m["ACC"]
far_delta = debate_m["FAR"] - baseline_m["FAR"]
frr_delta = debate_m["FRR"] - baseline_m["FRR"]

verdicts = []
if acc_delta > 0:   verdicts.append(f"accuracy improved by {acc_delta:.1f}pp")
elif acc_delta < 0: verdicts.append(f"accuracy degraded by {-acc_delta:.1f}pp")
else:               verdicts.append("accuracy unchanged")

if far_delta < 0:   verdicts.append(f"FAR reduced by {-far_delta:.1f}pp")
elif far_delta > 0: verdicts.append(f"FAR worsened by {far_delta:.1f}pp")

if frr_delta < 0:   verdicts.append(f"FRR reduced by {-frr_delta:.1f}pp")
elif frr_delta > 0: verdicts.append(f"FRR worsened by {frr_delta:.1f}pp")

print(f"\nVerdict: Debate {'IMPROVED' if acc_delta >= 0 and far_delta <= 0 else 'DID NOT IMPROVE'} over single-pass.")
print("  " + " | ".join(verdicts))

# ── Clause-level breakdown of changes ────────────────────────────────────────
n_changed     = df_cmp["impossible_changed"].sum()
n_improved    = ((df_cmp["impossible_changed"]) &
                 (df_cmp["final_impossible"] == df_cmp["is_impossible"])).sum()
n_degraded    = ((df_cmp["impossible_changed"]) &
                 (df_cmp["final_impossible"] != df_cmp["is_impossible"])).sum()

print(f"\nClause-level changes ({n_changed} total):")
print(f"  Improved (debate → correct): {n_improved}")
print(f"  Degraded (debate → wrong)  : {n_degraded}")
print(f"  Unchanged                  : {len(df_cmp) - n_changed}")


In [ ]:

# ── Export: merge debate columns onto input extraction dataframe ───────────────
#
# Produces one row per clause (same as df_selected), with these new columns:
#   debate_answer          – final_answer list after debate
#   debate_is_impossible   – final impossibility verdict
#   debate_changed         – True if debate changed the extraction
#   debate_consensus       – True if debate reached consensus (False = arbiter)
#   debate_stability       – stability score 0.0–1.0
#   debate_rounds          – debate rounds used
#   debate_reasoning       – final reasoning text
#
# Output files (timestamped):
#   <EXTRACTION_DIR>/debate_merged_<ts>.pkl   ← full dataframe with list columns
#   <EXTRACTION_DIR>/debate_merged_<ts>.csv   ← CSV with lists serialised as JSON
# ─────────────────────────────────────────────────────────────────────────────

_ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Build flat debate dataframe ───────────────────────────────────────────────
_debate_flat = _build_debate_df(all_debate_results)

# ── Left-join debate columns onto df_selected on (title, clause_name) ─────────
df_with_debate = df_selected.merge(
    _debate_flat,
    on=["title", "clause_name"],
    how="left",
)

# ── How many rows got debate results? ────────────────────────────────────────
_covered = df_with_debate["debate_answer"].notna().sum()
print(f"df_selected rows      : {len(df_selected)}")
print(f"Debate-covered rows   : {_covered}  ({_covered/len(df_selected)*100:.1f}%)")
print(f"Columns added         : {[c for c in df_with_debate.columns if c.startswith('debate_')]}")

# ── Save PKL (preserves list columns) ─────────────────────────────────────────
_pkl_out = EXTRACTION_DIR / f"debate_merged_{_ts}.pkl"
df_with_debate.to_pickle(_pkl_out)
print(f"\nSaved PKL → {_pkl_out}")

# ── Save CSV (lists serialised as JSON strings) ───────────────────────────────
_csv_out = EXTRACTION_DIR / f"debate_merged_{_ts}.csv"
_df_csv  = df_with_debate.copy()
for _col in ["debate_answer", "answers", "answer_ai"]:
    if _col in _df_csv.columns:
        _df_csv[_col] = _df_csv[_col].apply(
            lambda x: json.dumps(x) if isinstance(x, list) else x
        )
_df_csv.to_csv(_csv_out, index=False)
print(f"Saved CSV → {_csv_out}")

# ── Quick preview ─────────────────────────────────────────────────────────────
_preview_cols = [
    "clause_name", "is_impossible", "is_impossible_ai",
    "debate_is_impossible", "debate_changed", "debate_consensus", "debate_stability",
]
display(df_with_debate[[c for c in _preview_cols if c in df_with_debate.columns]].head(10))


In [ ]:
# Quick audit: what changed from initial -> debate (per contract + examples)

if 'df_debate' not in globals():
    raise ValueError('Run the debate summary cell first so df_debate is available.')

changed_mask = df_debate['answers_changed'] | df_debate['impossible_changed']
changed_df = df_debate[changed_mask].copy()

print('Per-contract change counts')
print('=' * 70)
per_contract = (
    changed_df.groupby('contract', as_index=False)
    .agg(changed_clauses=('clause_name', 'count'))
    .sort_values('changed_clauses', ascending=False)
)
print(per_contract.to_string(index=False))

print('\nTop changed clauses (sample)')
print('=' * 70)
sample_cols = [
    'contract', 'clause_name',
    'initial_impossible', 'final_impossible',
    'initial_n_answers', 'final_n_answers',
    'rounds_used', 'stability_score'
]
print(changed_df[sample_cols].head(20).to_string(index=False))

print('\nTotals')
print('=' * 70)
print(f"Contracts audited : {df_debate['contract'].nunique()}")
print(f"Clauses audited   : {len(df_debate)}")
print(f"Clauses changed   : {len(changed_df)}")
print(f"Answers changed   : {df_debate['answers_changed'].sum()}")
print(f"Impossible changed: {df_debate['impossible_changed'].sum()}")

In [ ]:
# Transcript diagnostics: where debate went wrong

import json

if 'all_debate_results' not in globals():
    raise ValueError('Run the debate batch cell first so all_debate_results is available.')
if 'df_selected' not in globals():
    raise ValueError('Run data/filter cells first so df_selected is available.')

# Build clause-level table with transcript fields
rows = []
for contract, clause_list in all_debate_results.items():
    for r in clause_list:
        hist = r.get('debate_history', []) or []
        sk_turns = [t for t in hist if t.get('role') == 'skeptic']
        su_turns = [t for t in hist if t.get('role') == 'supporter']

        sk_last = sk_turns[-1]['argument'] if sk_turns else {}
        su_last = su_turns[-1]['argument'] if su_turns else {}

        rows.append({
            'title': contract,
            'clause_name': r['clause_name'],
            'initial_answer': r['initial_answer'],
            'final_answer': r['final_answer'],
            'initial_is_impossible': r['initial_is_impossible'],
            'final_is_impossible': r['final_is_impossible'],
            'consensus_reached': r['consensus_reached'],
            'stability_score': r['stability_score'],
            'rounds_used': r['rounds_used'],
            'final_reasoning': r.get('final_reasoning', ''),
            'skeptic_verdict': sk_last.get('verdict'),
            'supporter_verdict': su_last.get('verdict'),
            'skeptic_quotes_n': len(sk_last.get('evidence_quotes', []) or []),
            'supporter_quotes_n': len(su_last.get('evidence_quotes', []) or []),
            'skeptic_challenge': sk_last.get('challenge', ''),
            'supporter_defense': su_last.get('defense', ''),
        })

transcript_df = pd.DataFrame(rows)

# Join GT + baseline output from df_selected
base_cols = ['title', 'clause_name', 'is_impossible', 'is_impossible_ai']
base = df_selected[base_cols].drop_duplicates(['title', 'clause_name'])
analysis_df = transcript_df.merge(base, on=['title', 'clause_name'], how='left')

# Correctness before/after debate (w.r.t. GT)
analysis_df['initial_correct'] = (analysis_df['is_impossible_ai'] == analysis_df['is_impossible'])
analysis_df['final_correct'] = (analysis_df['final_is_impossible'] == analysis_df['is_impossible'])
analysis_df['degraded'] = analysis_df['initial_correct'] & (~analysis_df['final_correct'])
analysis_df['improved'] = (~analysis_df['initial_correct']) & (analysis_df['final_correct'])
analysis_df['changed'] = (
    (analysis_df['initial_is_impossible'] != analysis_df['final_is_impossible']) |
    (analysis_df['initial_answer'].astype(str) != analysis_df['final_answer'].astype(str))
)

# Transition buckets
analysis_df['transition'] = analysis_df.apply(
    lambda x: (
        'present->absent' if (not x['initial_is_impossible'] and x['final_is_impossible']) else
        'absent->present' if (x['initial_is_impossible'] and not x['final_is_impossible']) else
        'present->present' if (not x['initial_is_impossible'] and not x['final_is_impossible']) else
        'absent->absent'
    ),
    axis=1
)

print('Failure Analysis Summary')
print('=' * 80)
print(f"Total clauses                 : {len(analysis_df)}")
print(f"Changed by debate             : {analysis_df['changed'].sum()}")
print(f"Degraded (correct -> wrong)   : {analysis_df['degraded'].sum()}")
print(f"Improved (wrong -> correct)   : {analysis_df['improved'].sum()}")
print()

print('Transitions among changed clauses')
print('-' * 80)
print(analysis_df[analysis_df['changed']]['transition'].value_counts().to_string())
print()

print('Verdict patterns in degraded cases')
print('-' * 80)
if analysis_df['degraded'].sum() > 0:
    vtab = (
        analysis_df[analysis_df['degraded']]
        .groupby(['skeptic_verdict', 'supporter_verdict'])
        .size()
        .sort_values(ascending=False)
    )
    print(vtab.to_string())
else:
    print('No degraded cases found.')
print()

print('Evidence quote coverage (degraded cases)')
print('-' * 80)
if analysis_df['degraded'].sum() > 0:
    d = analysis_df[analysis_df['degraded']]
    print(f"Avg skeptic quotes per clause  : {d['skeptic_quotes_n'].mean():.2f}")
    print(f"Avg supporter quotes per clause: {d['supporter_quotes_n'].mean():.2f}")
    print(f"No skeptic quotes              : {(d['skeptic_quotes_n'] == 0).sum()} / {len(d)}")
    print(f"No supporter quotes            : {(d['supporter_quotes_n'] == 0).sum()} / {len(d)}")
else:
    print('No degraded cases found.')
print()

print('Top degraded clause types/names')
print('-' * 80)
if analysis_df['degraded'].sum() > 0:
    print(analysis_df[analysis_df['degraded']]['clause_name'].value_counts().head(12).to_string())
else:
    print('No degraded cases found.')
print()

print('Sample degraded transcripts (first 8)')
print('-' * 80)
cols = [
    'title', 'clause_name', 'transition', 'rounds_used',
    'skeptic_verdict', 'supporter_verdict',
    'initial_is_impossible', 'final_is_impossible',
    'initial_answer', 'final_answer',
    'skeptic_challenge', 'supporter_defense'
]
show_df = analysis_df[analysis_df['degraded']][cols].head(8).copy()
for c in ['skeptic_challenge', 'supporter_defense']:
    show_df[c] = show_df[c].fillna('').astype(str).str.slice(0, 220)
print(show_df.to_string(index=False))


In [ ]:

# ============================================================
# DEEP TRANSCRIPT ROOT-CAUSE ANALYSIS
# Why is debate underperforming single-pass?
# ============================================================
import textwrap

# ── 1. Build per-clause records from all_debate_results ──────
#    all_debate_results: {title → [clause_result_dicts]}
#    clause_result_dict keys: clause_name, initial_answer, initial_is_impossible,
#      final_answer, final_is_impossible, final_reasoning, consensus_reached,
#      stability_score, rounds_used, debate_history, thread_id

gt_lookup = (
    df_selected[["title", "clause_name", "is_impossible", "is_impossible_ai"]]
    .drop_duplicates(["title", "clause_name"])
    .set_index(["title", "clause_name"])
)

records = []
for contract, clause_list in all_debate_results.items():
    for c in clause_list:
        key = (contract, c["clause_name"])
        if key not in gt_lookup.index:
            continue
        gt_row = gt_lookup.loc[key]
        gt_absent    = bool(gt_row["is_impossible"])
        base_absent  = bool(gt_row["is_impossible_ai"])
        final_absent = bool(c["final_is_impossible"])

        hist     = c.get("debate_history", []) or []
        sk_turns = [t for t in hist if t.get("role") == "skeptic"]
        su_turns = [t for t in hist if t.get("role") == "supporter"]
        sk = sk_turns[-1]["argument"] if sk_turns else {}
        su = su_turns[-1]["argument"] if su_turns else {}

        sk_v    = sk.get("verdict", "?")
        su_v    = su.get("verdict", "?")
        sk_conf = sk.get("confidence", "?")
        su_conf = su.get("confidence", "?")
        sk_q    = sk.get("evidence_quotes", []) or []
        su_q    = su.get("evidence_quotes", []) or []
        sk_flags = set(sk.get("scenario_flags", []) or [])
        su_flags = set(su.get("scenario_flags", []) or [])
        sk_spans = sk.get("unsupported_spans", []) or []
        su_conc  = su.get("concessions", []) or []

        base_correct  = (base_absent  == gt_absent)
        final_correct = (final_absent == gt_absent)
        degraded = base_correct and not final_correct
        improved = not base_correct and final_correct
        changed  = (base_absent != final_absent) or (
            str(c.get("initial_answer")) != str(c.get("final_answer"))
        )

        # ── Root-cause classification ────────────────────────
        if degraded:
            if not base_absent and final_absent:
                # A correct answer was REMOVED by debate → new FN
                if sk_v == "UNSUPPORTED" and len(sk_q) == 0:
                    root_cause = "RC1_skeptic_no_quotes_forced_removal"
                elif sk_v == "UNSUPPORTED" and su_v == "UNSUPPORTED":
                    root_cause = "RC2_both_UNSUPPORTED_agree_remove"
                elif su_v == "UNSUPPORTED" and len(su_conc) > 0:
                    root_cause = "RC3_supporter_capitulated_with_concessions"
                elif final_absent and sk_v in ("PARTIAL", "SUPPORTED"):
                    root_cause = "RC4_finalize_pruned_despite_weak_support"
                else:
                    root_cause = "RC5_present_to_absent_other"
            elif base_absent and not final_absent:
                # A wrong answer was ADDED to a GT-absent clause → new FP
                if su_v == "SUPPORTED" and len(su_q) >= 1:
                    root_cause = "RC6_supporter_hallucinated_new_answer"
                elif sk_v in ("PARTIAL", "SUPPORTED"):
                    root_cause = "RC7_skeptic_pushed_absent_to_present"
                else:
                    root_cause = "RC8_absent_to_present_other"
            else:
                root_cause = "RC9_answer_text_drifted"
        elif improved:
            root_cause = "IMPROVED"
        elif changed:
            root_cause = "neutral_change"
        else:
            root_cause = "unchanged"

        records.append({
            "contract": contract,
            "clause_name": c["clause_name"],
            "gt_absent": gt_absent,
            "base_absent": base_absent,
            "final_absent": final_absent,
            "base_correct": base_correct,
            "final_correct": final_correct,
            "degraded": degraded,
            "improved": improved,
            "changed": changed,
            "root_cause": root_cause,
            "rounds": c.get("rounds_used", 1),
            "sk_verdict": sk_v,
            "su_verdict": su_v,
            "sk_conf": sk_conf,
            "su_conf": su_conf,
            "sk_quotes_n": len(sk_q),
            "su_quotes_n": len(su_q),
            "sk_flags": sorted(sk_flags),
            "su_flags": sorted(su_flags),
            "sk_spans_n": len(sk_spans),
            "su_concessions_n": len(su_conc),
            "initial_answer": c.get("initial_answer"),
            "final_answer": c.get("final_answer"),
            "sk_challenge": str(sk.get("challenge", ""))[:350],
            "su_defense": str(su.get("defense", ""))[:350],
            "sk_quotes": sk_q[:3],
            "su_quotes": su_q[:3],
            "final_reasoning": str(c.get("final_reasoning", ""))[:350],
        })

df_rca = pd.DataFrame(records)

# ── 2. High-level counts ─────────────────────────────────────
SEP = "=" * 72
sep = "-" * 72
total_deg = int(df_rca["degraded"].sum())
print(SEP)
print("  TRANSCRIPT ROOT-CAUSE ANALYSIS")
print(SEP)
print(f"{'Total clauses':<35}: {len(df_rca)}")
print(f"{'Changed by debate':<35}: {df_rca['changed'].sum()}")
print(f"{'Degraded (correct → wrong)':<35}: {total_deg}")
print(f"{'Improved (wrong → correct)':<35}: {df_rca['improved'].sum()}")
print()

# ── 3. Root-cause frequency table ────────────────────────────
print(SEP)
print("  ROOT-CAUSE FREQUENCY  (degraded only)")
print(SEP)
rc_labels = {
    "RC1_skeptic_no_quotes_forced_removal":    "RC1  Skeptic UNSUPPORTED with 0 evidence quotes → answer dropped",
    "RC2_both_UNSUPPORTED_agree_remove":       "RC2  Both agents UNSUPPORTED → consensus delete",
    "RC3_supporter_capitulated_with_concessions":"RC3 Supporter conceded answer away with concessions",
    "RC4_finalize_pruned_despite_weak_support":"RC4  Finalize pruned despite only PARTIAL/SUPPORTED verdict",
    "RC5_present_to_absent_other":             "RC5  Present→absent, unclassified reason",
    "RC6_supporter_hallucinated_new_answer":   "RC6  Supporter added hallucinated answer (new FP)",
    "RC7_skeptic_pushed_absent_to_present":    "RC7  Skeptic pushed wrong absent→present",
    "RC8_absent_to_present_other":             "RC8  Absent→present, unclassified reason",
    "RC9_answer_text_drifted":                 "RC9  Answer text drifted, impossible flag unchanged",
}
rc_counts = (
    df_rca[df_rca["degraded"]]["root_cause"]
    .value_counts()
    .rename_axis("root_cause")
    .reset_index(name="count")
)
rc_map = dict(zip(rc_counts["root_cause"], rc_counts["count"]))
for _, row in rc_counts.iterrows():
    lbl = rc_labels.get(row["root_cause"], row["root_cause"])
    print(f"  [{row['count']:2d}/{total_deg}]  {lbl}")
print()

# ── 4. Verdict pairs in degraded cases ───────────────────────
print(SEP)
print("  VERDICT PAIRS IN DEGRADED CASES")
print(SEP)
if total_deg > 0:
    vp = (
        df_rca[df_rca["degraded"]]
        .groupby(["sk_verdict", "su_verdict"])
        .size()
        .sort_values(ascending=False)
    )
    print(vp.to_string())
else:
    print("  No degraded cases.")
print()

# ── 5. Evidence quality ───────────────────────────────────────
print(SEP)
print("  EVIDENCE QUALITY  (degraded vs improved)")
print(SEP)
for label, grp in [("degraded", df_rca[df_rca["degraded"]]), ("improved", df_rca[df_rca["improved"]])]:
    if len(grp) == 0:
        continue
    print(f"  {label.upper()} (n={len(grp)})")
    print(f"    avg SK quotes    : {grp['sk_quotes_n'].mean():.2f}  | zero-quote cases: {(grp['sk_quotes_n']==0).sum()}")
    print(f"    avg SU quotes    : {grp['su_quotes_n'].mean():.2f}  | zero-quote cases: {(grp['su_quotes_n']==0).sum()}")
    print(f"    avg SU concessions: {grp['su_concessions_n'].mean():.2f}")
    print(f"    avg rounds       : {grp['rounds'].mean():.2f}")
print()

# ── 6. Confidence in degraded cases ──────────────────────────
print(SEP)
print("  CONFIDENCE DISTRIBUTION  (degraded cases)")
print(SEP)
deg = df_rca[df_rca["degraded"]]
if len(deg) > 0:
    print("  Skeptic confidence:")
    print(deg["sk_conf"].value_counts().to_string())
    print("\n  Supporter confidence:")
    print(deg["su_conf"].value_counts().to_string())
print()

# ── 7. Full transcript per degraded case ─────────────────────
print(SEP)
print("  FULL TRANSCRIPT — EACH DEGRADED CASE")
print(SEP)
for idx, row in df_rca[df_rca["degraded"]].reset_index(drop=True).iterrows():
    transition = (
        "present→absent"  if not row["base_absent"] and row["final_absent"] else
        "absent→present"  if row["base_absent"] and not row["final_absent"]  else
        "present→present"
    )
    print(f"\n{'─'*72}")
    print(f"  [DEGRADED #{idx+1}]")
    print(f"  Contract    : {str(row['contract'])[:68]}")
    print(f"  Clause      : {row['clause_name']}")
    print(f"  Root cause  : {row['root_cause']}")
    print(f"  GT:base:fin absent  {row['gt_absent']} : {row['base_absent']} : {row['final_absent']}")
    print(f"  Transition  : {transition}   Rounds: {row['rounds']}")
    print(f"  SK verdict  : {row['sk_verdict']} ({row['sk_conf']})  quotes={row['sk_quotes_n']}  spans={row['sk_spans_n']}")
    print(f"  SU verdict  : {row['su_verdict']} ({row['su_conf']})  quotes={row['su_quotes_n']}  concessions={row['su_concessions_n']}")
    print(f"  SK flags    : {row['sk_flags']}")
    print(f"  SU flags    : {row['su_flags']}")
    print(f"  SK quotes   : {row['sk_quotes']}")
    print(f"  SU quotes   : {row['su_quotes']}")
    print("  SK challenge:")
    print(textwrap.fill(row["sk_challenge"], 76, initial_indent="    ", subsequent_indent="    "))
    print("  SU defense:")
    print(textwrap.fill(row["su_defense"], 76, initial_indent="    ", subsequent_indent="    "))
    print(f"  Initial ans : {str(row['initial_answer'])[:180]}")
    print(f"  Final ans   : {str(row['final_answer'])[:180]}")
    print("  Finalize reasoning:")
    print(textwrap.fill(row["final_reasoning"], 76, initial_indent="    ", subsequent_indent="    "))

# ── 8. Actionable findings ────────────────────────────────────
print()
print(SEP)
print("  ACTIONABLE FINDINGS")
print(SEP)
findings = []

rc1 = rc_map.get("RC1_skeptic_no_quotes_forced_removal", 0)
rc2 = rc_map.get("RC2_both_UNSUPPORTED_agree_remove", 0)
rc3 = rc_map.get("RC3_supporter_capitulated_with_concessions", 0)
rc4 = rc_map.get("RC4_finalize_pruned_despite_weak_support", 0)
rc6 = rc_map.get("RC6_supporter_hallucinated_new_answer", 0)
rc7 = rc_map.get("RC7_skeptic_pushed_absent_to_present", 0)

if rc1 > 0:
    findings.append(
        f"[FIX A] RC1 ({rc1} case{'s' if rc1>1 else ''}): Skeptic votes UNSUPPORTED with zero quotes. "
        "Rule → if SK evidence_quotes is empty, hard-cap verdict to PARTIAL; "
        "finalize then preserves the baseline answer."
    )
if rc2 > 0:
    findings.append(
        f"[FIX B] RC2 ({rc2} case{'s' if rc2>1 else ''}): Both agents reach UNSUPPORTED consensus and remove answer. "
        "Rule → require at least one direct contradicting quote from EACH agent before deletion."
    )
if rc3 > 0:
    findings.append(
        f"[FIX C] RC3 ({rc3} case{'s' if rc3>1 else ''}): Supporter conceded away a correct answer. "
        "Rule → if SU has concessions but zero supporting quotes, force SU verdict to PARTIAL "
        "and preserve baseline."
    )
if rc4 > 0:
    findings.append(
        f"[FIX D] RC4 ({rc4} case{'s' if rc4>1 else ''}): Finalize deleted answer despite non-UNSUPPORTED verdict. "
        "Rule → finalize only sets impossible=True when verdict==UNSUPPORTED AND both quote counts ≥1 "
        "AND no high-risk flags."
    )
if rc6 + rc7 > 0:
    findings.append(
        f"[FIX E] RC6+RC7 ({rc6+rc7} case{'s' if rc6+rc7>1 else ''}): Debate injected wrong answer into GT-absent clause. "
        "Rule → absent→present flip requires BOTH agents to give SUPPORTED verdict with ≥1 quote each."
    )
if not findings:
    findings.append(
        "No single dominant root cause. All degradation is diffuse. "
        "Consider running on more contracts to gather larger statistics."
    )

for ft in findings:
    print()
    print(textwrap.fill(ft, 80))

print()
print(f"  Baseline ACC: {baseline_m['ACC']:.1f}%  |  Debate ACC: {debate_m['ACC']:.1f}%  |  Delta: {acc_delta:+.1f}pp")
print(SEP)


In [ ]:

# ── v7.1 RCA: compact degradation summary ────────────────────────────────────
import re

rows_v7 = []
for contract_key, clause_list in all_debate_results.items():
    matched_title = None
    for t in df_selected["title"].unique():
        if t == contract_key: matched_title = t; break
    if matched_title is None:
        norm_k = re.sub(r'[^A-Z0-9]', '', contract_key.upper())
        for t in df_selected["title"].unique():
            if re.sub(r'[^A-Z0-9]', '', t.upper()) == norm_k: matched_title = t; break
    if matched_title is None: continue
    sub = (df_selected[df_selected["title"]==matched_title]
           [["clause_name","is_impossible"]].drop_duplicates("clause_name").set_index("clause_name"))
    for c in clause_list:
        cn = c["clause_name"]
        if cn not in sub.index: continue
        gt   = bool(sub.loc[cn, "is_impossible"])
        base = bool(c["initial_is_impossible"])
        fin  = bool(c["final_is_impossible"])
        hist = c.get("debate_history",[]) or []
        # Grab last SK/SU verdicts
        sk_turns = [t for t in hist if t.get("role")=="skeptic"]
        su_turns = [t for t in hist if t.get("role")=="supporter"]
        sk_v = sk_turns[-1]["argument"].get("verdict","?") if sk_turns else "?"
        su_v = su_turns[-1]["argument"].get("verdict","?") if su_turns else "?"
        sk_q = len(sk_turns[-1]["argument"].get("evidence_quotes",[]) or []) if sk_turns else 0
        su_q = len(su_turns[-1]["argument"].get("evidence_quotes",[]) or []) if su_turns else 0
        rows_v7.append(dict(
            cn=cn, gt=gt, base=base, fin=fin,
            degraded=(base==gt and fin!=gt),
            reext=bool(c.get("re_extraction_triggered", False)),
            sk_v=sk_v, su_v=su_v, sk_q=sk_q, su_q=su_q,
            reasoning=c.get("final_reasoning","")[:180],
        ))

deg7 = [r for r in rows_v7 if r["degraded"]]
print(f"Degraded: {len(deg7)}  Improved: {sum(1 for r in rows_v7 if r['base']!=r['gt'] and r['fin']==r['gt'])}")
print()
for r in deg7:
    direction = "present→absent" if not r["gt"] and r["fin"] else "absent→present"
    print(f"  [{direction}] {r['cn']}  reext={r['reext']}")
    print(f"    SK={r['sk_v']}(q={r['sk_q']})  SU={r['su_v']}(q={r['su_q']})")
    print(f"    {r['reasoning'][:160]}")
    print()
